# Inspector Liddy
### A murder. A manor. One methodical mind.

Sir Edmund Hartley is dead. Someone in this house is responsible.
Inspector Liddy will use the camera, his memory, and his considerable intellect to determine who.

Run each cell in order.

In [ ]:
# Install dependencies
!pip install ultralytics transformers torch opencv-python-headless matplotlib pillow --quiet

In [ ]:
import sys, os, json, random
from collections import deque, Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from PIL import Image
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import io

print('Imports ready.')

In [ ]:
# ---- Game world ----
# I went with Victorian names because Mr. Green felt too kindergarten

SUSPECTS = [
    'Lord Blackwood', 'Lady Ashford', 'Dr. Pembrooke',
    'Colonel Ravenswood', 'Miss Thorne', 'Graves the Butler'
]

WEAPONS = [
    'Candlestick', 'Letter Opener', 'Poison Vial',
    'Silk Cord', 'Revolver', 'Iron Poker'
]

ROOMS = [
    'Drawing Room', 'Library', 'Conservatory', 'Billiard Room',
    'Kitchen', 'Wine Cellar', 'Study', 'Ballroom', 'Servants Quarters'
]

YOLO_TO_WEAPON = {
    'knife': 'Letter Opener', 'scissors': 'Letter Opener',
    'bottle': 'Poison Vial', 'wine glass': 'Poison Vial',
    'cup': 'Candlestick', 'cell phone': 'Revolver',
    'remote': 'Revolver', 'umbrella': 'Iron Poker',
    'tie': 'Silk Cord',
}

# Generate the hidden mystery
mystery = {
    'culprit': random.choice(SUSPECTS),
    'weapon':  random.choice(WEAPONS),
    'room':    random.choice(ROOMS),
}
print('The mystery is set. Inspector Liddy does not yet know the answer either.')

In [ ]:
# ---- Inspector Liddy brain (in-notebook version) ----

probs = {
    'suspects': {s: 1.0 / len(SUSPECTS) for s in SUSPECTS},
    'weapons':  {w: 1.0 / len(WEAPONS)  for w in WEAPONS},
    'rooms':    {r: 1.0 / len(ROOMS)    for r in ROOMS},
}
memory      = deque(maxlen=10)
long_memory = []
thought_log = []
lidar_grid  = np.zeros((50, 50), dtype=np.float32)

def normalize(d):
    t = sum(d.values())
    return {k: v / t for k, v in d.items()} if t else d

def update_weapon(weapon, boost=2.0):
    if weapon in probs['weapons']:
        probs['weapons'][weapon] *= boost
        probs['weapons'] = normalize(probs['weapons'])

def update_suspects(count, boost=1.4):
    if count <= 0: return
    top_n = sorted(SUSPECTS, key=lambda s: probs['suspects'][s], reverse=True)[:count]
    for s in top_n:
        probs['suspects'][s] *= boost
    probs['suspects'] = normalize(probs['suspects'])

def top(category):
    return max(probs[category], key=lambda k: probs[category][k])

def confidence():
    return (max(probs['suspects'].values()) +
            max(probs['weapons'].values()) +
            max(probs['rooms'].values())) / 3

INTRO = [
    'Inspector Liddy. Called in because the local authorities are, as usual, insufficient.',
    'I observe. I remember. I deduce. In that order.',
    'Sir Edmund Hartley is dead. Someone in this house knows why.',
]
OBS_LINES = [
    'The {obj} is here. That is not nothing.',
    'A {obj} in plain sight. Either brazen or careless.',
    'One does not leave a {obj} about without reason.',
]
DED_LINES = [
    '{suspect} rises in my estimation. Not fondly.',
    'The evidence circles back to {suspect}.',
    'A {weapon} is not a weapon of impulse. This was deliberate.',
]

print(random.choice(INTRO))

In [ ]:
# ---- YOLO setup ----
from ultralytics import YOLO
yolo = YOLO('yolov8n.pt')
print('YOLO ready. Inspector Liddy has eyes.')

In [ ]:
# ---- Camera capture (Colab webcam) ----
# I went with the standard Colab JS snippet. It just works.

def capture_photo(filename='scan.jpg'):
    js = Javascript('''
        async function takePhoto(quality) {
            const div    = document.createElement('div');
            const btn    = document.createElement('button');
            btn.textContent = 'Capture';
            btn.style.margin = '10px';
            btn.style.padding = '8px 20px';
            const video  = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            div.appendChild(video);
            div.appendChild(btn);
            video.srcObject = stream;
            await video.play();
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            await new Promise(resolve => btn.onclick = resolve);
            const canvas = document.createElement('canvas');
            canvas.width  = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto(0.85)')
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

print('Camera ready. Run the next cell to scan a scene.')

In [ ]:
# ---- Scan the scene ---- Run this cell as many times as you like.

photo_path = capture_photo()
img_pil    = Image.open(photo_path).convert('RGB')
img_arr    = np.array(img_pil)

results    = yolo(img_arr, verbose=False)
detections = []
for r in results:
    for box in r.boxes:
        detections.append({
            'class':      yolo.names[int(box.cls)],
            'confidence': float(box.conf),
            'bbox':       box.xyxy[0].tolist(),
        })

# Annotate and display
import cv2
annotated = img_arr.copy()
for det in detections:
    x1, y1, x2, y2 = [int(v) for v in det['bbox']]
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (100, 220, 130), 2)
    cv2.putText(annotated, f"{det['class']} {det['confidence']:.2f}",
                (x1, max(y1-6,10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 220, 130), 1)

display(Image.fromarray(annotated))

# Feed to Inspector
weapons_found  = []
suspect_count  = sum(1 for d in detections if d['class'] == 'person')

for det in detections:
    mapped = YOLO_TO_WEAPON.get(det['class'])
    if mapped:
        weapons_found.append(mapped)
        update_weapon(mapped, boost=1.8 + det['confidence'])

if suspect_count:
    update_suspects(suspect_count)

memory.append({'weapon': weapons_found[0] if weapons_found else None,
               'suspect_count': suspect_count,
               'raw': [d['class'] for d in detections]})
long_memory.append(memory[-1])

# Update LIDAR
h, w = img_arr.shape[:2]
for det in detections:
    x1, y1, x2, y2 = det['bbox']
    gx = int(((x1+x2)/2/w) * 49)
    gy = int(((y1+y2)/2/h) * 49)
    lidar_grid[gy, gx] += det['confidence']
lidar_grid *= 0.97

# Inspector narrates
obj_label = weapons_found[0] if weapons_found else (detections[0]['class'] if detections else None)
if obj_label:
    narration = random.choice(OBS_LINES).format(obj=obj_label)
else:
    narration = 'Nothing of note in this frame. Continue.'

thought_log.append(narration)

if len(long_memory) % 3 == 0:
    ded_line = random.choice(DED_LINES).format(
        suspect=top('suspects'), weapon=top('weapons')
    )
    thought_log.append(ded_line)
    narration += ' ' + ded_line

print(f'Inspector Liddy: {narration}')
print(f'Detected: {[d["class"] for d in detections] or ["nothing"]}')
print(f'Leading suspect: {top("suspects")} | Confidence: {confidence():.1%}')

In [ ]:
# ---- LIDAR map ----

fig, ax = plt.subplots(figsize=(5, 5), facecolor='#111')
ax.set_facecolor('#111')
ax.imshow(lidar_grid, cmap='YlOrRd', origin='upper', vmin=0, vmax=max(lidar_grid.max(), 0.1))
ax.set_title('LIDAR Positional Scan', color='#c8a96e', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Forensics ---- Cross-reference what memory holds.

all_weapons = [e['weapon'] for e in long_memory if e.get('weapon')]
if all_weapons:
    most_common = Counter(all_weapons).most_common(1)[0][0]
    update_weapon(most_common, boost=1.6)
    print(f'Inspector Liddy: The {most_common} keeps surfacing. Pattern noted.')
else:
    print('Inspector Liddy: Insufficient weapon evidence for forensic cross-reference.')

In [ ]:
# ---- Interrogate a suspect ----

print('Available suspects:')
for i, s in enumerate(SUSPECTS):
    print(f'  {i}: {s}')

choice = int(input('Enter suspect number: '))
chosen = SUSPECTS[choice]
probs['suspects'][chosen] *= 2.0
probs['suspects'] = normalize(probs['suspects'])
print(f'Inspector Liddy: {chosen}. I have questions. They have answers they would prefer not to share.')

In [ ]:
# ---- Final accusation ----

accused_suspect = top('suspects')
accused_weapon  = top('weapons')
accused_room    = top('rooms')
conf            = confidence()

print(f'\nInspector Liddy:')
print(f'My conclusion is not a guess. It is a certainty built on evidence.')
print(f'{accused_suspect}, with the {accused_weapon}, in the {accused_room}.')
print(f'I am rarely wrong. I am not wrong now.')
print(f'\nConfidence: {conf:.1%}')

correct = (
    accused_suspect == mystery['culprit'] and
    accused_weapon  == mystery['weapon']  and
    accused_room    == mystery['room']
)

print(f'\n{"CORRECT. As expected." if correct else "WRONG. That is unexpected."}')
print(f'The truth: {mystery["culprit"]} / {mystery["weapon"]} / {mystery["room"]}')

In [ ]:
# ---- Export case report ----

report = {
    'inspector': 'Inspector Liddy',
    'mystery': mystery,
    'final_deduction': {
        'suspect': accused_suspect,
        'weapon':  accused_weapon,
        'room':    accused_room,
        'confidence': round(conf, 4),
        'correct': correct,
    },
    'thought_log': thought_log,
    'evidence': list(long_memory),
}

with open('inspector_liddy_case.json', 'w') as f:
    json.dump(report, f, indent=2)

print('Case report written to inspector_liddy_case.json')

# Download it
from google.colab import files
files.download('inspector_liddy_case.json')